<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_1/notebooks/Unit_1_Data_Analysis_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 1 Data Analysis Assignment — Colocalization Analysis of a Confocal Image

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

You have one confocal image (a `.czi` file from a Zeiss microscope) of a brain section stained with two
antibodies and a nuclear counterstain. Each antibody was visualised with a different fluorescent dye,
and the microscope recorded each dye as a separate **channel**:

| Channel | Antibody | What that antibody binds |
|---|---|---|
| DAPI | — | DNA, so every nucleus in the field |
| **Alexa 488** | anti-**GSTπ** | GSTπ, a protein that appears only once an oligodendrocyte has **matured** and begun myelinating |
| **Alexa 594** | anti-**Olig2** | Olig2, a transcription factor present in **every cell of the oligodendrocyte lineage**, mature or not |

From here on each channel is named by its dye and its antibody together — **Alexa 488 (GSTπ)** and
**Alexa 594 (Olig2)** — because you need both: the dye is what the microscope recorded, and the antibody
is what it means.

Your task is to count the cells labelled in each channel, and then find the cells where **both** dyes
appear in the same place. Finding two signals in the same spot is called **colocalization**.

**Run every cell in order, from the top** (click a cell, press **Shift + Enter**). The **💬 Discussion** at the end is the part you write.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Set up and load the image [1 pt]

This installs `czifile` (which reads Zeiss `.czi` files) and downloads the image from the course
GitHub. The download only happens once.

In [ ]:
!pip install czifile imagecodecs --quiet

import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from czifile import CziFile
from skimage.measure import label, regionprops

IMAGE_FILE = "Image_2.czi"
URL = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_1/images/" + IMAGE_FILE

if not os.path.exists(IMAGE_FILE):
    print("Downloading", IMAGE_FILE, "...")
    try:
        urllib.request.urlretrieve(URL, IMAGE_FILE)
    except Exception:
        print("\n*** The download did not work. ***")
        print("Click the folder icon in the left sidebar, upload " + IMAGE_FILE + ",")
        print("then run this cell again.")
        raise SystemExit

print("Ready.", IMAGE_FILE, "is", round(os.path.getsize(IMAGE_FILE) / 1e6, 1), "MB")

## Step 2 — Look at the image [1 pt]

Before measuring anything, look at what you're measuring.

The cell below opens the file and pulls out the three channels. It also reads the **pixel size** from
the metadata the microscope stored inside the file — that is what lets us report results in real units
(µm and mm²) rather than pixels.

In [ ]:
# Open a .czi file and return the three channels plus the pixel size in microns
def open_image(path):
    with CziFile(path) as czi:
        # The pixel size is stored in the metadata, in metres. Convert to microns.
        metadata = czi.metadata()
        x_size = float(re.search(r'Distance Id="X">\s*<Value>([^<]+)<', metadata).group(1)) * 1e6

        data = np.squeeze(czi.asarray())     # drop empty dimensions -> (channel, y, x)

    # This microscope saved the channels in the order Alexa 488, DAPI, Alexa 594.
    channels = {"Alexa488": data[0], "DAPI": data[1], "Alexa594": data[2]}
    return channels, x_size


channels, pixel_microns = open_image(IMAGE_FILE)

height, width = channels["DAPI"].shape
area_mm2 = (height * width * pixel_microns * pixel_microns) / 1e6

print("Image size:", width, "x", height, "pixels")
print("One pixel is", round(pixel_microns, 3), "microns across")
print("Total imaged area:", round(area_mm2, 4), "mm²")

In [ ]:
plt.figure(figsize=(15, 4.5))

for position, (name, title) in enumerate([("DAPI", "DAPI — all nuclei"),
                                          ("Alexa594", "Alexa 594"),
                                          ("Alexa488", "Alexa 488")]):
    plt.subplot(1, 3, position + 1)
    plt.imshow(channels[name], cmap="gray", vmax=np.percentile(channels[name], 99.5))
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

Now all three at once, as a colour **composite**. Alexa 594 (Olig2) is displayed in red, Alexa 488
(GSTπ) in green, and DAPI in blue. Where the two dyes land in the same place, the mix shows up
**yellow**.

The colours are a display choice, not a property of the dyes — any channel could be shown in any
colour.

In [ ]:
def stretch(image):
    low, high = np.percentile(image, 1), np.percentile(image, 99.7)
    return np.clip((image - low) / (high - low), 0, 1)


composite = np.dstack([stretch(channels["Alexa594"]),   # red
                       stretch(channels["Alexa488"]),   # green
                       stretch(channels["DAPI"])])      # blue

plt.figure(figsize=(8, 8))
plt.imshow(composite)
plt.title("Composite:  red = Alexa 594    green = Alexa 488    blue = DAPI")
plt.axis("off")
plt.show()

## Step 3 — Detecting the cells [2 pt]

A computer doesn't see "cells" — it sees a grid of brightness values. Three steps turn one into the
other:

1. **Threshold** — keep only pixels brighter than a cut-off. This decides what counts as real staining.
2. **Size filter** — drop blobs too small to be a cell (noise) or too large (several merged together).
3. **Label** — give every remaining blob its own number, so we can count them.

Below the code you'll see each raw channel next to what the computer decided was a cell.

> **Look carefully at those "detected cells" panels.** Whatever they show — a lot, a little, or nothing
> at all — that is a **result**, not an error. Write down the numbers and keep going; the Discussion
> asks you about them.

In [ ]:
# brightness cut-offs. A pixel counts as signal only if it is brighter than this.
THRESHOLD_594 = 400
THRESHOLD_488 = 800

# Size filter, in pixels: each object should be about the size of one cell body
MIN_SIZE, MAX_SIZE = 100, 3000


# Turn one channel into a mask of detected cells, plus a numbered version of it
def find_cells(image, threshold, min_size, max_size):
    mask = image > threshold                       # 1. threshold

    labelled = label(mask)                         # 2. number every separate blob

    # 3. keep only blobs whose size is in range
    good_blobs = [blob.label for blob in regionprops(labelled)
                  if min_size <= blob.area <= max_size]
    keep = np.isin(labelled, good_blobs)

    return keep, label(keep)


mask_594, labels_594 = find_cells(channels["Alexa594"], THRESHOLD_594, MIN_SIZE, MAX_SIZE)
mask_488, labels_488 = find_cells(channels["Alexa488"], THRESHOLD_488, MIN_SIZE, MAX_SIZE)

print("Alexa 594+ cells detected:", labels_594.max())
print("Alexa 488+ cells detected:", labels_488.max())

In [ ]:
plt.figure(figsize=(11, 9))

for position, (name, title, mask) in enumerate([("Alexa594", "Alexa 594", mask_594),
                                                ("Alexa488", "Alexa 488", mask_488)]):
    plt.subplot(2, 2, position * 2 + 1)
    plt.imshow(channels[name], cmap="gray", vmax=np.percentile(channels[name], 99.5))
    plt.title(title + " — raw")
    plt.axis("off")

    plt.subplot(2, 2, position * 2 + 2)
    plt.imshow(mask, cmap="gray")
    plt.title(title + " — detected cells")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 4 — Colocalization and your result [2 pts]

A pixel counts as **colocalized** only if it belongs to a detected cell in **both** channels at once.
We label those overlap regions and count them — these are the **double-positive** cells, the ones that
appeared yellow in the composite.

Then we turn counts into results. A raw count depends on how much tissue you photographed, so we divide
by the imaged area to get a **density** in cells per mm². We also compute the **colocalization ratio** — what proportion of the Alexa 594⁺ cells were
also Alexa 488⁺ :

$$\text{colocalization ratio} = \frac{\text{Alexa 488}^{+}\text{Alexa 594}^{+}\text{ cells}}{\text{Alexa 594}^{+}\text{ cells}}$$

In [ ]:
# A pixel is colocalized only if it was detected in BOTH channels
coloc_mask = np.logical_and(labels_594 > 0, labels_488 > 0)
coloc_labels = label(coloc_mask)
coloc_count = coloc_labels.max()

plt.figure(figsize=(13, 4.5))
for position, (mask, title) in enumerate([(mask_594, "Alexa 594+ "),
                                          (mask_488, "Alexa 488+ "),
                                          (coloc_mask, "Overlap = double-positive")]):
    plt.subplot(1, 3, position + 1)
    plt.imshow(mask, cmap="gray")
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Where they sit in the real image
plt.figure(figsize=(8, 8))
plt.imshow(composite)
for blob in regionprops(coloc_labels):
    y, x = blob.centroid
    plt.scatter(x, y, s=900, facecolors="none", edgecolors="white", linewidths=2)
plt.title(str(coloc_count) + " double-positive cells (circled)")
plt.axis("off")
plt.show()

In [ ]:
count_594 = labels_594.max()
count_488 = labels_488.max()

# If nothing was detected we cannot compute a fraction, so record it as "not available"
if count_594 > 0:
    fraction = round(100 * coloc_count / count_594, 1)
else:
    fraction = np.nan

my_result = pd.DataFrame([{
    "Filename": IMAGE_FILE,
    "THRESHOLD_594": THRESHOLD_594,
    "THRESHOLD_488": THRESHOLD_488,
    "Alexa594_Cells": count_594,
    "Alexa488_Cells": count_488,
    "Colocalized_Cells": coloc_count,
    "Image_Area_mm2": round(area_mm2, 4),
    "Alexa594_per_mm2": round(count_594 / area_mm2, 1),
    "Alexa488_per_mm2": round(count_488 / area_mm2, 1),
    "Colocalized_cells_per_mm2": round(coloc_count / area_mm2, 1),
    "Colocalization_ratio_pct": fraction,
}])

my_result.to_csv("my_image_result.csv", index=False)
print("Saved my_image_result.csv\n")

my_result.T          # .T flips the table on its side so it is easier to read

### 💬 Discussion

**Questions 1 need you to change a setting and re-run.** Change `THRESHOLD_594` and
`THRESHOLD_488` in Step 3, then re-run Step 3 and Step 4 to get each set of numbers.

1. The initial threashold for was set to **400 and 800**, but now change the thresholds to **4000 and 8000** and re-run Steps 3 and 4. How many cells do you see now, and why do you think there's a difference? [1 pt]?

2. Given what each antibody
   binds (see the table at the top introduction), what does cells **expressing Alexa 594 positive and Alexa 488 negative** tell you about that cell's
   identity [1 pt]?

3. Given what each antibody
   binds (see the table at the top), what does a cell **expressing Alexa 594 positive** and **Alexa 488 positive** tell you about that cell's
   identity [1 pt]?

4. Report the **colocalization ratio** from the step 4 and explain what it means **biologically** [1 pt].



**Your answer:** *(double-click here to type)*

>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.